# AttnFool evaluation

Evaluates the seven classifiers used in the paper on a 1000-image subset of
ImageNet-2012 val, plus DETR on a 100-image subset of MS COCO 2017 val.

Classifiers: **ResNet50, ViT-T, ViT-B, ViT-B/384, DeiT-T, DeiT-B, DeiT-B/384**.

Before running, build the data subsets:

```bash
python -m datasets.prepare_data --imagenet-src /path/to/ImageNet/val
```

Knobs to tune cost: `NUM_IMAGENET`, `NUM_COCO`, `ATTACK_ITERS`, `BATCH_SIZE`,
`MODELS_TO_RUN`.

In [1]:
%pip install numpy
%pip install matplotlib
%pip install scipy
%pip install scikit-learn
%pip install torch
%pip install torchvision
%pip install torchaudio
%pip install transformers
%pip install datasets
%pip install wandb
%pip install timm
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os, sys, math, time, json, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image

REPO = Path('.').resolve()
sys.path.insert(0, str(REPO))

from models.DeiT import (
    deit_tiny_patch16_224, deit_base_patch16_224, deit_base_patch16_384,
)
from models.vision_transformer import (
    vit_tiny_patch16_224, vit_base_patch16_224, vit_base_patch16_384,
)
from models.resnet import ResNet50
from utils import (
    apply_patch, attn_fool_loss, build_patch_mask, cosine_step_size,
    normalized_momentum, patch_token_index, mu as IMNET_MU, std as IMNET_STD,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

ModuleNotFoundError: No module named 'models.DeiT'

## Config

In [ ]:
NUM_IMAGENET = 1000          # images sampled by datasets.prepare_data
NUM_COCO     = 100

BATCH_SIZE   = 16            # drop if you OOM on the 384 models
ATTACK_ITERS = 250           # paper default; lower to e.g. 50 for a quick run
ATTACK_LR    = 8.0 / 255.0
ATTACK_MODE  = 'AttnFool_kq' # 'CE_loss' | 'AttnFool_kq' | 'AttnFool_kqstar'
ATTN_W       = 1.0
USE_MOMENTUM = False
PATCH_SIZE   = 16
PATCH_ROW    = 0
PATCH_COL    = 0
SEED         = 0

MODELS_TO_RUN = [
    'ResNet50',
    'ViT-T', 'ViT-B', 'ViT-B-384',
    'DeiT-T', 'DeiT-B', 'DeiT-B-384',
]

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

## Model registry

Each entry returns `(model, is_transformer, img_size)`. The transformer
flag controls whether we pull `(q,k)` from the model for the AttnFool
loss.

In [ ]:
def _resnet50():
    m = ResNet50(pretrained=True)
    return m, False, 224

MODEL_FACTORY = {
    'ResNet50'  : _resnet50,
    'ViT-T'     : lambda: (vit_tiny_patch16_224(pretrained=True), True, 224),
    'ViT-B'     : lambda: (vit_base_patch16_224(pretrained=True), True, 224),
    'ViT-B-384' : lambda: (vit_base_patch16_384(pretrained=True), True, 384),
    'DeiT-T'    : lambda: (deit_tiny_patch16_224(pretrained=True), True, 224),
    'DeiT-B'    : lambda: (deit_base_patch16_224(pretrained=True), True, 224),
    'DeiT-B-384': lambda: (deit_base_patch16_384(pretrained=True), True, 384),
}

def forward(model, x, is_transformer):
    if is_transformer:
        out, qk_list = model(x)
        return out, qk_list
    return model(x), None

## ImageNet val loader (1000 images)

In [ ]:
IMAGENET_DIR = REPO / 'data' / 'imagenet_val'
assert IMAGENET_DIR.is_dir(), (
    f'Missing {IMAGENET_DIR}. Run `python -m datasets.prepare_data --imagenet-src <path>` first.'
)

def make_imagenet_loader(img_size, batch_size=BATCH_SIZE):
    resize = int(img_size / 0.875) if img_size == 224 else img_size
    tfm = transforms.Compose([
        transforms.Resize(resize, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMNET_MU, std=IMNET_STD),
    ])
    ds = torchvision.datasets.ImageFolder(str(IMAGENET_DIR), transform=tfm)
    print(f'  imagenet({img_size}): {len(ds)} images, {len(ds.classes)} classes')
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)

## AttnFool attack (per batch)

Mirrors `main.py`. The patch is initialized uniformly in `[0,1]` at
`(PATCH_ROW, PATCH_COL)` and refined with PGD on cosine-decayed step size.

In [ ]:
def attack_batch(model, is_transformer, X, y, network_name,
                 attack_mode=ATTACK_MODE, n_iters=ATTACK_ITERS,
                 lr=ATTACK_LR, atten_w=ATTN_W, use_momentum=USE_MOMENTUM,
                 patch_size=PATCH_SIZE, patch_row=PATCH_ROW, patch_col=PATCH_COL,
                 img_size=224):
    mu_t  = torch.tensor(IMNET_MU).view(1, 3, 1, 1).to(X.device)
    std_t = torch.tensor(IMNET_STD).view(1, 3, 1, 1).to(X.device)
    criterion = nn.CrossEntropyLoss()
    mask = build_patch_mask(X.shape, patch_size, patch_row, patch_col, X.device)

    target_token_idx = (
        patch_token_index(network_name, patch_row, patch_col,
                          patch_size=patch_size, img_size=img_size)
        if is_transformer else 0
    )

    x_01 = X * std_t + mu_t
    patch_01 = torch.rand_like(X) * mask + x_01 * (1 - mask)
    patch_01 = patch_01.detach().requires_grad_(True)
    m_state = torch.zeros_like(patch_01) if use_momentum else None

    for t in range(n_iters):
        if patch_01.grad is not None:
            patch_01.grad = None
        x_adv = apply_patch(X, patch_01, mask, mu_t, std_t)
        out, qk_list = forward(model, x_adv, is_transformer)
        loss = criterion(out, y)
        if is_transformer and attack_mode == 'AttnFool_kq':
            loss = loss + atten_w * attn_fool_loss(qk_list, target_token_idx, None)
        elif is_transformer and attack_mode == 'AttnFool_kqstar':
            loss = loss + atten_w * attn_fool_loss(qk_list, target_token_idx, 0)
        grad = torch.autograd.grad(loss, patch_01)[0]

        if use_momentum:
            m_state = normalized_momentum(m_state, grad, beta=0.9)
            direction = m_state.sign()
        else:
            direction = grad.sign()

        alpha_t = cosine_step_size(lr, t, n_iters)
        with torch.no_grad():
            patch_01 = patch_01 + alpha_t * direction * mask
            patch_01 = patch_01.clamp(0, 1)
            patch_01 = patch_01 * mask + x_01 * (1 - mask)
        patch_01.requires_grad_(True)

    with torch.no_grad():
        x_adv = apply_patch(X, patch_01, mask, mu_t, std_t)
        out, _ = forward(model, x_adv, is_transformer)
    return out

## Run the classifier evaluation

In [ ]:
import subprocess, os
from pathlib import Path

RESULTS_PATH = Path('results.json')
results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}

def run_models(names):
    """Run each model in its own subprocess so CUDA memory is fully released
    on process exit. Merges results into `results` and persists results.json
    after each model so a crash mid-group doesn't lose earlier wins."""
    for name in names:
        print(f'\n=== {name} ===', flush=True)
        cmd = [
            sys.executable, '-u', 'run_model.py',
            '--name', name,
            '--num-imagenet', str(NUM_IMAGENET),
            '--batch-size', str(BATCH_SIZE),
            '--attack-iters', str(ATTACK_ITERS),
            '--attack-lr', str(ATTACK_LR),
            '--attack-mode', ATTACK_MODE,
            '--attn-w', str(ATTN_W),
            '--patch-size', str(PATCH_SIZE),
            '--patch-row', str(PATCH_ROW),
            '--patch-col', str(PATCH_COL),
            '--seed', str(SEED),
        ]
        if USE_MOMENTUM:
            cmd.append('--use-momentum')

        env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, cwd=str(REPO), bufsize=1)
        last_json = None
        for line in proc.stdout:
            line = line.rstrip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and 'clean_acc' in obj:
                    last_json = obj
                    continue
            except json.JSONDecodeError:
                pass
            print(line, flush=True)
        proc.wait()
        if proc.returncode != 0 or last_json is None:
            print(f'  [{name}] FAILED (exit {proc.returncode})', flush=True)
            continue
        results[name] = {'clean_acc': last_json['clean_acc'],
                         'attnfool_acc': last_json['attnfool_acc']}
        RESULTS_PATH.write_text(json.dumps(results, indent=2))
    return results


### Group 1: ResNet50, ViT-T, ViT-B (224)

In [ ]:
run_models(['ResNet50', 'ViT-T', 'ViT-B'])
print(json.dumps(results, indent=2))

### Group 2: DeiT-T, DeiT-B (224)

In [ ]:
run_models(['DeiT-T', 'DeiT-B'])
print(json.dumps(results, indent=2))

### Group 3: ViT-B-384, DeiT-B-384 (384, heaviest)

In [ ]:
run_models(['ViT-B-384', 'DeiT-B-384'])
print('\nSummary'); print(json.dumps(results, indent=2))

## DETR on COCO val subset

In [ ]:
from models.detr import build_detr, detect, COCO_CLASSES

COCO_DIR = REPO / 'datasets' / 'coco_val'
assert COCO_DIR.is_dir(), 'Run datasets/prepare_data.py to populate datasets/coco_val first.'

detr = build_detr('detr_resnet50', pretrained=True).to(DEVICE).eval()

img_paths = sorted((COCO_DIR / 'images').glob('*.jpg'))[:NUM_COCO]
print(f'COCO subset: {len(img_paths)} images')

with open(COCO_DIR / 'instances_val2017_subset.json') as f:
    gt = json.load(f)
gt_by_file = {im['file_name']: im for im in gt['images']}
gt_per_img = {}
for a in gt['annotations']:
    gt_per_img.setdefault(a['image_id'], []).append(a)

per_image = []
for p in img_paths:
    img = Image.open(p).convert('RGB')
    dets = detect(detr, img, score_threshold=0.7, device=DEVICE)
    gt_meta = gt_by_file.get(p.name, {})
    per_image.append({
        'file'    : p.name,
        'n_dets'  : len(dets),
        'n_gt'    : len(gt_per_img.get(gt_meta.get('id'), [])),
        'labels'  : sorted({d['label'] for d in dets}),
    })

avg_dets = sum(r['n_dets'] for r in per_image) / max(1, len(per_image))
avg_gt   = sum(r['n_gt']   for r in per_image) / max(1, len(per_image))
print(f'\nAvg DETR detections / image (score>0.7): {avg_dets:.2f}')
print(f'Avg ground-truth boxes / image          : {avg_gt:.2f}')
for r in per_image[:5]:
    print(r)

### Visualize one DETR prediction

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

img = Image.open(img_paths[0]).convert('RGB')
dets = detect(detr, img, score_threshold=0.7, device=DEVICE)
fig, ax = plt.subplots(figsize=(8, 8)); ax.imshow(img); ax.axis('off')
for d in dets:
    x0, y0, x1, y1 = d['box_xyxy']
    ax.add_patch(mpatches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=2, edgecolor='lime'))
    ax.text(x0, y0, f"{d['label']} {d['score']:.2f}", color='black',
            bbox=dict(facecolor='lime', alpha=0.6, pad=1))
plt.show()